In [4]:
import os
from dotenv import load_dotenv
load_dotenv() # aloading all the enviroment variable

groq_api_key = os.getenv("GROQ_API_KEY")


In [5]:
from langchain_groq import ChatGroq

model = ChatGroq(model="groq/compound",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7f8df09ddfa0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f8df09e56a0>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi I am rohit and I am a System engineer ")])

AIMessage(content='Hi Rohit! 👋 Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 254, 'total_tokens': 332, 'completion_time': 0.175657, 'completion_tokens_details': None, 'prompt_time': 0.010546, 'prompt_tokens_details': None, 'queue_time': 0.168906, 'total_time': 0.186202}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019e6794-fb71-7781-bf5f-04dfc75230b4-0', usage_metadata={'input_tokens': 254, 'output_tokens': 78, 'total_tokens': 332})

In [7]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi I am rohit and I am a System engineer"),
        AIMessage(content="Hi Rohit! Nice to meet you. How can I assist you today?"),
        HumanMessage(content="Who am I? What is my name?")
    ]
)

AIMessage(content='**Answer**\n\nYou are **Rohit**, and you work as a **System Engineer**.\n\n---\n\n### Reasoning\n\n1. **Initial Introduction**  \n   - In your first message you wrote: “Hi I am rohit and I am a System engineer.”  \n   - This explicitly gives both your name (**Rohit**) and your profession (**System engineer**).\n\n2. **Clarification Request**  \n   - You later asked, “Who am I? What is my name?”  \n   - The only personal information you have provided so far is the name “Rohit” and the role “System engineer.”\n\n3. **Conclusion**  \n   - Therefore, based on the information you supplied, the answer to “Who am I?” is that you are a **System Engineer** named **Rohit**.', additional_kwargs={'reasoning_content': '\nTo answer the question "Who am I? What is my name?", I need to recall the information provided earlier in the conversation.\n\nThe user introduced themselves as "Rohit" and mentioned their profession as a "System engineer".\n\n\nMy name is Rohit, and I am a Syste

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keepp track of inputs and outputs of the model, and store them in some datastore.
Future interaction will then load those messages and pass them into the chain as part of the input.

In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    for key,val in store.items():
        
        print(key,"->",val)
        
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

/Users/rohit/Desktop/LLM&HuggingFace/languageTranslation/.venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3550: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [9]:
config = {"configurable":{"session_id":"chat1"}}

In [10]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Rohit and I am a System Engineer")],
    config=config
)

chat1 -> 


In [11]:
response.content

'Hello Rohit! Nice to meet you. How can I assist you today?'

In [12]:
response = with_message_history.invoke(
    [HumanMessage(content="Who am I?")],
    config=config
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?


In [13]:
response.content

'You’re Rohit, and you work as a system engineer.'

In [15]:
config1 = {"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="Who am I")],
    config=config1
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Who am I?
AI: You’re Rohit, and you work as a system engineer.
chat2 -> 


In [16]:
response.content

'I’m afraid I can’t determine who you are.\u202fBased on our conversation so far you haven’t shared any identifying details—no name, background, interests, or other personal information—so there isn’t enough data for me to infer your identity. If you’d like me to address you in a particular way or discuss something specific about yourself, feel free to let me know!'

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content="My name is Jhon the Don")],
    config=config1
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Who am I?
AI: You’re Rohit, and you work as a system engineer.
chat2 -> Human: Who am I
AI: I’m afraid I can’t determine who you are. Based on our conversation so far you haven’t shared any identifying details—no name, background, interests, or other personal information—so there isn’t enough data for me to infer your identity. If you’d like me to address you in a particular way or discuss something specific about yourself, feel free to let me know!


In [18]:
response = with_message_history.invoke(
    [HumanMessage(content="Who am I")],
    config=config1
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Who am I?
AI: You’re Rohit, and you work as a system engineer.
chat2 -> Human: Who am I
AI: I’m afraid I can’t determine who you are. Based on our conversation so far you haven’t shared any identifying details—no name, background, interests, or other personal information—so there isn’t enough data for me to infer your identity. If you’d like me to address you in a particular way or discuss something specific about yourself, feel free to let me know!
Human: My name is Jhon the Don
AI: Nice to meet you, Jhon the Don! Let me know if there’s anything you’d like to talk about or ask.


In [19]:
response.content

'You’re\u202fJhon\u202fthe\u202fDon. If there’s anything specific you’d like to discuss or any way I can help you, just let me know!'

# Prompt Template
Prompt Template help to turn raw user information into a format that the LLM can work with. In case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [20]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer the questions to the next of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [21]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Rohit")]})

AIMessage(content='Hello Rohit! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 279, 'total_tokens': 341, 'completion_time': 0.130849, 'completion_tokens_details': None, 'prompt_time': 0.011008, 'prompt_tokens_details': None, 'queue_time': 0.170912, 'total_time': 0.141857}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019e6796-b0db-70d0-a3f9-1b54787624b5-0', usage_metadata={'input_tokens': 279, 'output_tokens': 62, 'total_tokens': 341})

In [22]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

/Users/rohit/Desktop/LLM&HuggingFace/languageTranslation/.venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3550: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [23]:
config={"configurable":{"session_id":"chat3"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hi my name is Rohit")],
    config=config
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Who am I?
AI: You’re Rohit, and you work as a system engineer.
chat2 -> Human: Who am I
AI: I’m afraid I can’t determine who you are. Based on our conversation so far you haven’t shared any identifying details—no name, background, interests, or other personal information—so there isn’t enough data for me to infer your identity. If you’d like me to address you in a particular way or discuss something specific about yourself, feel free to let me know!
Human: My name is Jhon the Don
AI: Nice to meet you, Jhon the Don! Let me know if there’s anything you’d like to talk about or ask.
Human: Who am I
AI: You’re Jhon the Don. If there’s anything specific you’d like to discuss or any way I can help you, just let me know!
chat3 -> 


In [24]:
response.content

'Hi Rohit! Nice to meet you. How can I help you today?'

In [25]:
# Adding more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all the question to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt|model

In [26]:
response = chain.invoke({"messages":[HumanMessage(content="Hi My name is Rohit")],"language":"Hindi"})

In [27]:
response.content

'नमस्ते रोहित! आपसे मिलकर ख़ुशी हुई। आपका दिन शुभ हो!'

Let's now wrap this more complicated chain in a message history class. This time because there are multiple keys input, we need to specify the correct key to use to save the chat history.

In [28]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

/Users/rohit/Desktop/LLM&HuggingFace/languageTranslation/.venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3550: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [29]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="Hi I am rohit")],"language":'Hindi'},
    config=config
)


chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Who am I?
AI: You’re Rohit, and you work as a system engineer.
chat2 -> Human: Who am I
AI: I’m afraid I can’t determine who you are. Based on our conversation so far you haven’t shared any identifying details—no name, background, interests, or other personal information—so there isn’t enough data for me to infer your identity. If you’d like me to address you in a particular way or discuss something specific about yourself, feel free to let me know!
Human: My name is Jhon the Don
AI: Nice to meet you, Jhon the Don! Let me know if there’s anything you’d like to talk about or ask.
Human: Who am I
AI: You’re Jhon the Don. If there’s anything specific you’d like to discuss or any way I can help you, just let me know!
chat3 -> Human: Hi my name is Rohit
AI: Hi Rohit! Nice to meet you. How can I help you today?
chat4 -> 


In [30]:
response.content

'नमस्ते रोहित!  \n\nआपने अपना परिचय दिया था – “Hi I am rohit” – और मैं इसे एक साधारण अभिवादन के रूप में समझा। इस जानकारी के आधार पर मैंने आपको एक मित्रवत उत्तर दिया, जिसमें मैंने अपना परिचय (मैं एक AI सहायक हूँ) और यह बताया कि मैं आपकी किस प्रकार मदद कर सकता हूँ।  \n\nयदि आपके पास कोई प्रश्न, समस्या, या कोई विशेष जानकारी चाहिए, तो कृपया बताइए—मैं यहाँ हिंदी में आपकी सहायता करने के लिए तैयार हूँ।'

In [32]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="What is my name?")],"language":"hindi"},
    config=config
)
response.content

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Who am I?
AI: You’re Rohit, and you work as a system engineer.
chat2 -> Human: Who am I
AI: I’m afraid I can’t determine who you are. Based on our conversation so far you haven’t shared any identifying details—no name, background, interests, or other personal information—so there isn’t enough data for me to infer your identity. If you’d like me to address you in a particular way or discuss something specific about yourself, feel free to let me know!
Human: My name is Jhon the Don
AI: Nice to meet you, Jhon the Don! Let me know if there’s anything you’d like to talk about or ask.
Human: Who am I
AI: You’re Jhon the Don. If there’s anything specific you’d like to discuss or any way I can help you, just let me know!
chat3 -> Human: Hi my name is Rohit
AI: Hi Rohit! Nice to meet you. How can I help you today?
chat4 -> Human: Hi I am rohit
AI: नमस्ते रोहित!  

'आपका नाम रोहित है।'

## Manage the conversation history

One of the important concept of understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.